# Ejercicio 1.2 · comparar tres modelos sobre el mismo corpus

Vamos a embedear el mismo corpus con tres modelos y comparar resultados:

| Modelo | Tipo |
|---|---|
| `paraphrase-multilingual-MiniLM-L12-v2` | open-source, local, gratis |
| Cohere `embed-multilingual-v3.0` | API con free tier |
| OpenAI `text-embedding-3-small` | API barata |

**Objetivo:** no decidir quién «gana», sino ver las fortalezas de cada uno con pruebas concretas.

## Paso 1 · Claves y dependencias

Las librerías ya están en `pyproject.toml` (se añadieron con `uv add`), así que no hay que instalar nada.
Las claves están en `.env`. Esta celda solo comprueba que existen, **sin imprimirlas**.

In [1]:
import os
import time

import numpy as np
import pandas as pd
from dotenv import load_dotenv

# Lee el fichero .env y mete las claves en las variables de entorno
load_dotenv()

print("OpenAI:", os.getenv("OPENAI_API_KEY") is not None)
print("Cohere:", os.getenv("COHERE_API_KEY") is not None)

OpenAI: True
Cohere: True


## Paso 2 · Corpus de 100 textos

Usamos `ag_news` (noticias en inglés con 4 categorías). Nos quedamos con 100 textos y sus etiquetas
(las etiquetas nos servirán en el Bonus 2).

In [2]:
from datasets import load_dataset

# Descarga solo los 100 primeros ejemplos del split de test
dataset = load_dataset("fancyzhx/ag_news", split="test[:100]")

# En ag_news las etiquetas son números: 0, 1, 2, 3. Aquí está su nombre
nombres_categorias = dataset.features["label"].names
print(nombres_categorias)

# Separamos en dos listas paralelas: textos[i] tiene la categoría etiquetas[i]
textos = []
etiquetas = []
for ejemplo in dataset:
    textos.append(ejemplo["text"])
    etiquetas.append(nombres_categorias[ejemplo["label"]])

print(len(textos), "textos")
print(textos[0])
print(etiquetas[0])

['World', 'Sports', 'Business', 'Sci/Tech']
100 textos
Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.
Business


## Paso 3 · Embeddings con los tres modelos

Guardamos todo en dos diccionarios para comparar después:
- `embeddings["nombre_modelo"]` → matriz de 100 filas (una por texto)
- `tiempos["nombre_modelo"]` → segundos que tardó

In [3]:
embeddings = {}
tiempos = {}

### Modelo 1 · MiniLM (local)

In [4]:
from sentence_transformers import SentenceTransformer

modelo_local = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# time.time() devuelve la hora actual en segundos; restando dos medimos la duración
inicio = time.time()
vectores_local = modelo_local.encode(textos)
fin = time.time()

embeddings["minilm"] = vectores_local
tiempos["minilm"] = fin - inicio
print("Tamaño del vector:", vectores_local.shape)

W1116 12:24:08.385000 49415 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Tamaño del vector: (100, 384)


### Modelo 2 · Cohere `embed-multilingual-v3.0`

Pistas:
- Cliente: `cohere.ClientV2(api_key=...)`
- Método: `cliente.embed(texts=..., model=..., input_type=..., embedding_types=["float"])`
- `input_type` es obligatorio: para documentos es `"search_document"` y para consultas `"search_query"`.
- El free tier limita las llamadas: envía los textos en lotes (máx. 96 por llamada).
- Los vectores salen en `respuesta.embeddings.float_`.

In [5]:
import cohere

cliente_cohere = cohere.ClientV2(api_key=os.getenv("COHERE_API_KEY"))

inicio = time.time()

# Cohere acepta máximo 96 textos por llamada, así que vamos por lotes
vectores_cohere = []
tamano_lote = 96
for posicion in range(0, len(textos), tamano_lote):
    lote = textos[posicion : posicion + tamano_lote]
    respuesta = cliente_cohere.embed(
        texts=lote,
        model="embed-multilingual-v3.0",
        input_type="search_document",  # son documentos que luego se buscan
        embedding_types=["float"],
    )
    # extend añade todos los vectores del lote a la lista grande
    vectores_cohere.extend(respuesta.embeddings.float_)

fin = time.time()

embeddings["cohere"] = np.array(vectores_cohere)
tiempos["cohere"] = fin - inicio
print("Tamaño del vector:", embeddings["cohere"].shape)

Tamaño del vector: (100, 1024)


### Modelo 3 · OpenAI `text-embedding-3-small`

Pistas:
- Cliente: `OpenAI()` (lee `OPENAI_API_KEY` solo).
- Método: `cliente.embeddings.create(model=..., input=lista_de_textos)`.
- Cada vector está en `respuesta.data[i].embedding`.

In [6]:
from openai import OpenAI

cliente_openai = OpenAI()

inicio = time.time()
respuesta = cliente_openai.embeddings.create(
    model="text-embedding-3-small",
    input=textos,
)
fin = time.time()

# Un vector por cada texto, en el mismo orden
vectores_openai = []
for elemento in respuesta.data:
    vectores_openai.append(elemento.embedding)

embeddings["openai"] = np.array(vectores_openai)
tiempos["openai"] = fin - inicio

# OpenAI nos dice cuántos tokens ha contado de verdad (lo usamos en el Paso 7)
tokens_reales_openai = respuesta.usage.total_tokens
print("Tamaño del vector:", embeddings["openai"].shape)
print("Tokens reales de los 100 textos:", tokens_reales_openai)

Tamaño del vector: (100, 1536)
Tokens reales de los 100 textos: 5863


### Resumen: tiempo y tamaño de cada modelo

In [7]:
filas_resumen = []
for nombre_modelo in embeddings:
    fila = {
        "modelo": nombre_modelo,
        "tiempo_segundos": round(tiempos[nombre_modelo], 2),
        "dimensiones": embeddings[nombre_modelo].shape[1],
    }
    filas_resumen.append(fila)

tabla_resumen = pd.DataFrame(filas_resumen)
tabla_resumen

,modelo,tiempo_segundos,dimensiones
0,minilm,0.61,384
1,cohere,0.93,1024
2,openai,2.81,1536


## Paso 4 · Set de 10 queries

Cada query mide algo distinto (el corpus está en inglés, así que las queries en español miden la capacidad multilingüe):

- **Literales (2):** palabras que aparecen tal cual en el corpus.
- **Sinónimos (3):** mismo significado, otras palabras.
- **Otro idioma (3):** en español.
- **Ambiguas (2):** una sola palabra con varios sentidos posibles.

In [8]:
queries = [
    # Literales: las palabras salen tal cual en varios textos
    "Chavez recall vote",
    "dollar euro exchange rate",
    # Sinónimos: no comparten palabras con los textos
    "spacecraft exploring other planets",     # Cassini, Mars Express, Ganymede
    "criminals stealing money online",        # phishing, card fraud
    "chip maker postpones new product",       # Intel delays
    # Otro idioma: en español sobre un corpus en inglés
    "elecciones en Venezuela",
    "medalla de oro en natación",
    "ataque de hackers a ordenadores",
    # Ambiguas: una palabra con varios sentidos
    "Apple",                                  # empresa, o fruta
    "Johnson",                                # Randy Johnson (béisbol), otros...
]
print(len(queries), "queries")

10 queries


## Paso 5 · Top-5 por query y por modelo

Cosas a tener en cuenta:
- La query se embebe **con el mismo modelo** que el corpus.
- Cohere: la query lleva `input_type="search_query"`.
- Similitud coseno = `vector_corpus @ vector_query` solo si los vectores están normalizados (mira el Bonus 1).
  Por eso la función tiene un parámetro `normalizar`: primero lo dejamos en `False` (producto escalar tal cual) y en el Bonus 1 en `True`.
- `np.argsort(valores)` devuelve los índices ordenados de menor a mayor.

In [9]:
# Guardamos los vectores de las queries ya calculados para no repetir llamadas a la API
vectores_de_queries = {}


def embedear_query(nombre_modelo, query):
    clave = (nombre_modelo, query)
    if clave in vectores_de_queries:
        return vectores_de_queries[clave]

    if nombre_modelo == "minilm":
        vector = modelo_local.encode([query])[0]
    elif nombre_modelo == "cohere":
        respuesta = cliente_cohere.embed(
            texts=[query],
            model="embed-multilingual-v3.0",
            input_type="search_query",  # aquí es una consulta, no un documento
            embedding_types=["float"],
        )
        vector = np.array(respuesta.embeddings.float_[0])
    else:
        respuesta = cliente_openai.embeddings.create(
            model="text-embedding-3-small",
            input=[query],
        )
        vector = np.array(respuesta.data[0].embedding)

    vectores_de_queries[clave] = vector
    return vector


def buscar_top5(nombre_modelo, query, normalizar=False):
    matriz = embeddings[nombre_modelo]
    vector_query = embedear_query(nombre_modelo, query)

    if normalizar:
        # Dividir cada vector entre su norma lo deja con longitud 1
        normas = np.linalg.norm(matriz, axis=1, keepdims=True)
        matriz = matriz / normas
        vector_query = vector_query / np.linalg.norm(vector_query)

    # @ multiplica la matriz por el vector: una similitud por cada texto del corpus
    similitudes = matriz @ vector_query

    # argsort ordena de menor a mayor; [::-1] lo da la vuelta y [:5] se queda con 5
    indices_ordenados = np.argsort(similitudes)[::-1]
    top5_indices = indices_ordenados[:5]
    top5_scores = similitudes[top5_indices]
    return top5_indices, top5_scores

In [10]:
nombres_modelos = ["minilm", "cohere", "openai"]
filas_top5 = []

for query in queries:
    for nombre_modelo in nombres_modelos:
        indices, scores = buscar_top5(nombre_modelo, query)
        fila = {
            "query": query,
            "modelo": nombre_modelo,
            "top5_indices": list(indices),
            "score_top1": round(float(scores[0]), 3),
            "texto_top1": textos[indices[0]][:60],
            "categoria_top1": etiquetas[indices[0]],
        }
        filas_top5.append(fila)

tabla_top5 = pd.DataFrame(filas_top5)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 250)
tabla_top5[["query", "modelo", "score_top1", "categoria_top1", "texto_top1"]]

,query,modelo,score_top1,categoria_top1,texto_top1
0,Chavez recall vote,minilm,17.485,World,Venezuela Prepares for Chavez Recall Vote Supporters and riv
1,Chavez recall vote,cohere,0.609,World,"Venezuelans Flood Polls, Voting Extended CARACAS, Venezuela"
2,Chavez recall vote,openai,0.663,World,Venezuela Prepares for Chavez Recall Vote Supporters and riv
3,dollar euro exchange rate,minilm,13.367,Business,Dollar Rises Vs Euro on Asset Flows Data NEW YORK (Reuters)
4,dollar euro exchange rate,cohere,0.665,Business,Dollar Briefly Hits 4-Wk Low Vs Euro LONDON (Reuters) - The
5,dollar euro exchange rate,openai,0.550,Business,Dollar Briefly Hits 4-Wk Low Vs Euro LONDON (Reuters) - The
6,spacecraft exploring other planets,minilm,14.759,Sci/Tech,Cassini Spies Two Little Saturn Moons (AP) AP - NASA's Cassi
7,spacecraft exploring other planets,cohere,0.508,Sci/Tech,Cassini Spies Two Little Saturn Moons (AP) AP - NASA's Cassi
8,spacecraft exploring other planets,openai,0.420,Sci/Tech,Mars Rovers Relay Images Through Mars Express European Space
9,criminals stealing money online,minilm,9.206,Sci/Tech,"Card fraud unit nets 36,000 cards In its first two years, th"


In [11]:

# Vemos los 5 textos de cada modelo para cada query (recortados a 60 caracteres)
for query in queries:
    print("=" * 100)
    print("QUERY:", query)
    for nombre_modelo in nombres_modelos:
        indices, scores = buscar_top5(nombre_modelo, query)
        print("  --", nombre_modelo)
        for posicion in range(5):
            indice = indices[posicion]
            print("     ", round(float(scores[posicion]), 3), "[", indice, "]", textos[indice][:60])


QUERY: Chavez recall vote
  -- minilm
      17.485 [ 36 ] Venezuela Prepares for Chavez Recall Vote Supporters and riv
      17.059 [ 84 ] Venezuela vote keeps Chavez as president CARACAS -- Venezuel
      15.5 [ 46 ] Venezuelans Flood Polls, Voting Extended  CARACAS, Venezuela
      7.16 [ 42 ] Politics an Afterthought Amid Hurricane (AP) AP - If Hurrica
      4.956 [ 52 ] Restive Maldives eases curfew after rounding up dissidents (
  -- cohere
      0.609 [ 46 ] Venezuelans Flood Polls, Voting Extended  CARACAS, Venezuela
      0.573 [ 84 ] Venezuela vote keeps Chavez as president CARACAS -- Venezuel
      0.559 [ 36 ] Venezuela Prepares for Chavez Recall Vote Supporters and riv
      0.348 [ 57 ] Delegates Urge Cleric to Pull Out of Najaf BAGHDAD, Iraq - D
      0.328 [ 80 ] Kerry Widens Lead in California, Poll Finds (Reuters) Reuter
  -- openai
      0.663 [ 36 ] Venezuela Prepares for Chavez Recall Vote Supporters and riv
      0.639 [ 46 ] Venezuelans Flood Polls, Voting Extende

In [12]:

# ¿Cuántos textos comparten los modelos en su top-5? (0 a 5)
# set() es una colección sin repetidos; & calcula los elementos que están en los dos
filas_solape = []
for query in queries:
    top_minilm = set(buscar_top5("minilm", query)[0])
    top_cohere = set(buscar_top5("cohere", query)[0])
    top_openai = set(buscar_top5("openai", query)[0])
    fila = {
        "query": query,
        "minilm_cohere": len(top_minilm & top_cohere),
        "minilm_openai": len(top_minilm & top_openai),
        "cohere_openai": len(top_cohere & top_openai),
        "los_tres": len(top_minilm & top_cohere & top_openai),
    }
    filas_solape.append(fila)

tabla_solape = pd.DataFrame(filas_solape)
print(tabla_solape)
print()
print(tabla_solape[["minilm_cohere", "minilm_openai", "cohere_openai", "los_tres"]].mean())


                                query  minilm_cohere  minilm_openai  cohere_openai  los_tres
0                  Chavez recall vote              3              3              5         3
1           dollar euro exchange rate              4              4              4         4
2  spacecraft exploring other planets              4              4              5         4
3     criminals stealing money online              4              4              4         4
4    chip maker postpones new product              4              4              4         3
5             elecciones en Venezuela              4              5              4         4
6          medalla de oro en natación              4              4              5         4
7     ataque de hackers a ordenadores              3              3              4         3
8                               Apple              3              3              3         3
9                             Johnson              2              2   

## Paso 6 · Análisis de divergencias

*(se rellena tras ver los resultados)*

## Paso 7 · Coste para 1 millón de documentos

Fórmula: `coste = (tokens_totales / 1_000_000) * precio_por_millón_de_tokens`

Comprueba los precios actuales en las webs oficiales de Cohere y OpenAI (cambian con el tiempo, no los des por buenos de memoria).
MiniLM es local, así que su coste en API es 0.

Para estimar los tokens: una aproximación habitual es ~1 token por cada 4 caracteres en inglés.

In [13]:
# --- 1) Estimación con la regla de ~4 caracteres por token ---
suma_caracteres = 0
for texto in textos:
    suma_caracteres = suma_caracteres + len(texto)
caracteres_medios = suma_caracteres / len(textos)
tokens_medios_estimados = caracteres_medios / 4
tokens_estimados_por_millon_de_docs = tokens_medios_estimados * 1_000_000

# --- 2) Dato real: OpenAI nos contó los tokens de los 100 textos ---
tokens_medios_reales = tokens_reales_openai / len(textos)
tokens_reales_por_millon_de_docs = tokens_medios_reales * 1_000_000

print("Caracteres medios por texto:", round(caracteres_medios))
print("Tokens medios (estimados con /4):", round(tokens_medios_estimados))
print("Tokens medios (reales, OpenAI):  ", round(tokens_medios_reales))

# Precios en USD por millón de tokens.
# OpenAI: confirmado en la página oficial del modelo (developers.openai.com).
# Cohere: NO he podido confirmarlo en su web oficial; 0.10 es el valor que recuerdo
#         para embed v3. Compruébalo en cohere.com/pricing y cámbialo si es distinto.
precio_cohere_por_millon_tokens = 0.10
precio_openai_por_millon_tokens = 0.02

# Usamos los tokens reales (más fiables que la regla de /4)
# Ojo: Cohere cuenta los tokens con su propio tokenizador, así que es una aproximación.
coste_cohere = (tokens_reales_por_millon_de_docs / 1_000_000) * precio_cohere_por_millon_tokens
coste_openai = (tokens_reales_por_millon_de_docs / 1_000_000) * precio_openai_por_millon_tokens

print()
print("Coste de 1 millón de documentos:")
print("  MiniLM (local): 0 USD en API (pero hay que pagar tu máquina/GPU)")
print("  Cohere:", round(coste_cohere, 2), "USD")
print("  OpenAI:", round(coste_openai, 2), "USD")
print("Diferencia Cohere - OpenAI:", round(coste_cohere - coste_openai, 2), "USD")

Caracteres medios por texto: 267
Tokens medios (estimados con /4): 67
Tokens medios (reales, OpenAI):   59

Coste de 1 millón de documentos:
  MiniLM (local): 0 USD en API (pero hay que pagar tu máquina/GPU)
  Cohere: 5.86 USD
  OpenAI: 1.17 USD
Diferencia Cohere - OpenAI: 4.69 USD


## Bonus 1 · Normalización

¿Devuelve cada modelo vectores con norma 1? Pista: `np.linalg.norm(vector)` calcula la norma.
Si no lo están, normaliza con `v / np.linalg.norm(v)` y compara resultados antes y después.

In [14]:
for nombre_modelo in nombres_modelos:
    matriz = embeddings[nombre_modelo]
    normas_primeros = []
    for numero in range(5):
        norma = np.linalg.norm(matriz[numero])
        normas_primeros.append(round(float(norma), 3))
    print(nombre_modelo, "-> normas de los 5 primeros vectores:", normas_primeros)

minilm -> normas de los 5 primeros vectores: [4.086, 3.778, 3.849, 3.595, 3.802]
cohere -> normas de los 5 primeros vectores: [1.0, 1.0, 1.0, 1.0, 1.0]
openai -> normas de los 5 primeros vectores: [1.0, 1.0, 1.0, 1.0, 1.0]


In [15]:

# Repetimos todas las búsquedas con normalizar=True y comparamos el top-5 con el de antes
filas_normalizacion = []
for nombre_modelo in nombres_modelos:
    queries_que_cambian = 0
    for query in queries:
        top_antes = list(buscar_top5(nombre_modelo, query, normalizar=False)[0])
        top_despues = list(buscar_top5(nombre_modelo, query, normalizar=True)[0])
        if top_antes != top_despues:
            queries_que_cambian = queries_que_cambian + 1
    filas_normalizacion.append({"modelo": nombre_modelo, "queries_con_top5_distinto": queries_que_cambian})

pd.DataFrame(filas_normalizacion)


,modelo,queries_con_top5_distinto
0,minilm,7
1,cohere,0
2,openai,1


## Bonus 2 · Calidad cuantitativa (zero-shot)

Idea: para cada texto, compara su embedding con el embedding de cada nombre de categoría
(`nombres_categorias`) y elige la categoría con mayor similitud. Compara con `etiquetas` y calcula el % de aciertos por modelo.

In [16]:
filas_zero_shot = []

for nombre_modelo in nombres_modelos:
    # Matriz de textos normalizada (así @ da similitud coseno de verdad)
    matriz = embeddings[nombre_modelo]
    normas = np.linalg.norm(matriz, axis=1, keepdims=True)
    matriz_normalizada = matriz / normas

    # Una columna de similitudes por cada categoría
    similitudes_por_categoria = []
    for nombre_categoria in nombres_categorias:
        vector_categoria = embedear_query(nombre_modelo, nombre_categoria)
        vector_categoria = vector_categoria / np.linalg.norm(vector_categoria)
        similitudes_por_categoria.append(matriz_normalizada @ vector_categoria)

    # Para cada texto elegimos la categoría con mayor similitud
    aciertos = 0
    for numero_texto in range(len(textos)):
        mejor_categoria = None
        mejor_similitud = -1
        for numero_categoria in range(len(nombres_categorias)):
            similitud = similitudes_por_categoria[numero_categoria][numero_texto]
            if similitud > mejor_similitud:
                mejor_similitud = similitud
                mejor_categoria = nombres_categorias[numero_categoria]
        if mejor_categoria == etiquetas[numero_texto]:
            aciertos = aciertos + 1

    filas_zero_shot.append({"modelo": nombre_modelo, "aciertos": aciertos, "porcentaje": aciertos})

pd.DataFrame(filas_zero_shot)

,modelo,aciertos,porcentaje
0,minilm,62,62
1,cohere,54,54
2,openai,60,60


## Conclusiones

*(¿Qué modelo elegirías para qué caso y por qué? Basa la respuesta en lo que has medido.)*